In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os 
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [6]:
df = pd.read_csv("/kaggle/input/english-crime-data/AnnotatedNewsEnglish.csv")
new_df = pd.read_csv("/kaggle/input/english-crime-data/ModelLabedlStudentVerified1.csv", encoding="ISO-8859-1")
news_df_correct = pd.read_csv("/kaggle/input/english-crime-data/RecordsThatWasLabeldCorrectly.csv", encoding="ISO-8859-1")
news_df_incorrect = pd.read_csv("/kaggle/input/english-crime-data/RecordsWasNotLabeledCorrectly.csv")
new_df.rename(columns = {"predicted_labels": "Label", "news content": "Content", "title": "Title", "date": "Date" }, inplace=True)
news_df_correct.rename(columns = {"news content": "Content", "title": "Title" }, inplace=True)
news_df_incorrect.rename(columns = {"news content": "Content", "title": "Title" }, inplace=True)
new_df.drop(columns=['ID'], inplace=True) 
news_df_correct.drop(columns=['_id'], inplace=True) 
news_df_incorrect.drop(columns=['_id'], inplace=True) 


In [7]:
news_df_correct['arr_labels']= news_df_correct[["final_labels[0]", "final_labels[1]"]].apply(
    lambda x: [val for val in x if pd.notna(val)], axis=1
)
news_df_incorrect['arr_labels']= news_df_correct[["final_labels[0]", "final_labels[1]"]].apply(
    lambda x: [val for val in x if pd.notna(val)], axis=1
)

In [ ]:
news_df_correct['arr_labels']

In [ ]:
news_df_correct.shape

In [10]:
df['Label'] = df['Label'].apply(lambda x: x.lower())
other_df =  df[df['Label'] == "other"]
cat_df =  df[df['Label'] != "other"]

sampled_df = pd.concat([cat_df, other_df.sample(5200)])
sampled_df.shape
sampled_df['Label']
sampled_df = pd.concat([sampled_df, new_df])

sampled_df['arr_labels'] = sampled_df['Label'].str.split(",").map(lambda x: [lbl.strip() for lbl in x] if isinstance(x, list) else ['other'])

sampled_df = sampled_df.sample(sampled_df.shape[0])


In [11]:
sampled_df['text'] = sampled_df['Title'] + sampled_df['Content']

In [12]:
sampled_df

,Date,URL,Title,Content,Label,arr_labels,text
9538,5/4/2023,https://english.aaj.tv/news/30320057/suspect-a...,Suspect arrested in deadly Atlanta hospital sh...,WASHINGTON: Police have arrested a man suspect...,murder,[murder],Suspect arrested in deadly Atlanta hospital sh...
13345,26/06/2023,https://tribune.com.pk/story/2423588/man-gunne...,Man gunned down during burglary,As many as 11 suspects burgled two houses and ...,murder,[murder],Man gunned down during burglaryAs many as 11 s...
7944,4/11/2022,https://tribune.com.pk/story/2384571/political...,Political assassinations: have we learnt any l...,"October 16, 1951, stands out in the history of...",other,[other],Political assassinations: have we learnt any l...
3397,2/12/2015,https://tribune.com.pk/story/837145/two-childr...,'Toy bomb' kills two children near LoC in Azad...,A bomb planted in a toy exploded near the Line...,other,[other],'Toy bomb' kills two children near LoC in Azad...
409,3/22/2014,https://english.aaj.tv/news/10258545/three-Mur...,Three Murdered for 'honour' in Sargodha,SARGODHA: Three persons including a woman were...,murder,[murder],Three Murdered for 'honour' in SargodhaSARGODH...
...,...,...,...,...,...,...,...
4202,7/1/2015,https://tribune.com.pk/story/912541/vice-polic...,Vice police: Vigilante shot dead,A man was shot dead in Burma Town after he and...,murder,[murder],Vice police: Vigilante shot deadA man was ...
19403,11/25/2024,https://tribune.com.pk/story/2511677/city-grap...,"City grapples with violence, tragedies",A series of grim events over the past week has...,murder,[murder],"City grapples with violence, tragediesA series..."
11163,12/21/2023,https://english.aaj.tv/news/30344966/,Pakistan demands UN investigation into source ...,Pakistan has demanded that the United Nations ...,other,[other],Pakistan demands UN investigation into source ...
8880,1/10/2022,https://www.nation.com.pk/E-Paper/gwadar/2022-...,Copy of controversial cipher missing from PM H...,Cabinet forms committee to investigate and fix...,other,[other],Copy of controversial cipher missing from PM H...


In [ ]:
sampled_df['arr_labels'].value_counts()

In [9]:
# import pandas as pd

# priority = ["robbery", "suicide", "kidnapping", "terrorist attack", 
#             "sexual assault", "murder", "other"]

# # Create mapping from category → priority rank
# priority_map = {cat: i for i, cat in enumerate(priority)}

# def multi_to_single(label_list):
#     # If no label present, assign "other"
#     if not label_list:
#         return "other"
#     # Pick label with highest priority (lowest index in priority_map)
#     return min(label_list, key=lambda x: priority_map.get(x, len(priority)))

# # Apply directly on arr_labels
# sampled_df['single_class'] = sampled_df['arr_labels'].apply(multi_to_single)

# print(sampled_df[['arr_labels', 'single_class']].sample(100))


In [10]:
# # Map single_class → label ids
# sampled_df['labels'] = sampled_df['single_class'].map(cat2id)

# # Drop rows with 'none' or any unmapped categories
# sampled_df = sampled_df.dropna(subset=['labels'])

# # Ensure integer dtype
# sampled_df['labels'] = sampled_df['labels'].astype(int)


In [ ]:
# ======================
# 1️⃣ Imports
# ======================
import pandas as pd
import torch
import torch.nn as nn
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.preprocessing import MultiLabelBinarizer
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

# ======================
# 2️⃣ Categories & MultiLabelBinarizer
# ======================
categories = ["robbery", "suicide", "kidnapping", "terrorist attack", 
              "sexual assault", "murder", "other"]
mlb = MultiLabelBinarizer(classes=categories)

# ✅ Ensure arr_labels is list of labels
# Example row: ["robbery","murder"] → [1,0,0,0,0,1,0]
sampled_df["labels"] = mlb.fit_transform(sampled_df["arr_labels"]).tolist()
sampled_df = sampled_df.drop_duplicates(subset=["URL"], keep="first").reset_index(drop=True)

news_df_correct['labels'] = mlb.transform(news_df_correct["arr_labels"]).tolist()
news_df_incorrect['labels'] = mlb.transform(news_df_incorrect["arr_labels"]).tolist()

merge_to = pd.concat([news_df_correct, news_df_incorrect])


sampled_df = sampled_df[~sampled_df["URL"].isin(merge_to["URL"])].reset_index(drop=True)


num_labels = len(categories)

# ======================
# 3️⃣ Train-Test Split
# ======================
train_df, test_df = train_test_split(sampled_df, test_size=0.2, random_state=42, shuffle=True)

test_df = pd.concat([test_df, news_df_correct.iloc[1400:] ])
train_df = pd.concat([news_df_incorrect, train_df, news_df_correct.iloc[:1400] ])
train_df, val_df = train_test_split(train_df, test_size=0.1, random_state=42, shuffle=True)


def clean_invalid_text(df):
    df = df.dropna(subset=["text"])   # remove NaN
    df = df[df["text"].apply(lambda x: isinstance(x, str))]  # keep only strings
    df = df[df["text"].str.strip() != ""]  # remove empty strings
    return df.reset_index(drop=True)

# Apply to all splits
train_df = clean_invalid_text(train_df)
val_df   = clean_invalid_text(val_df)
test_df  = clean_invalid_text(test_df)

print(f"Testing Data: {test_df.shape}\n Training Data: {train_df.shape}\n   Validation Data: {val_df.shape}")


train_dataset = Dataset.from_pandas(train_df[['text','labels']], preserve_index=False)
val_dataset   = Dataset.from_pandas(val_df[['text','labels']], preserve_index=False)
test_dataset  = Dataset.from_pandas(test_df[['text','labels']], preserve_index=False)


# ======================
# 3️⃣ Tokenizer + Model
# ======================
MODEL_ID = "google-bert/bert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    num_labels=num_labels,
    problem_type="multi_label_classification",  # ✅ important
    ignore_mismatched_sizes=True
)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

def tokenize(batch):
    return tokenizer(batch['text'], padding='max_length', truncation=True, max_length=512)

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset   = val_dataset.map(tokenize, batched=True)
test_dataset  = test_dataset.map(tokenize, batched=True)

train_dataset.set_format(type='torch', columns=['input_ids','attention_mask','labels'])
val_dataset.set_format(type='torch', columns=['input_ids','attention_mask','labels'])
test_dataset.set_format(type='torch', columns=['input_ids','attention_mask','labels'])

# ======================
# 4️⃣ Multi-Label Focal Loss
# ======================
class MultiLabelFocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, reduction="mean"):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
        self.bce = nn.BCEWithLogitsLoss(reduction="none")

    def forward(self, logits, targets):
        bce_loss = self.bce(logits, targets.float())
        pt = torch.exp(-bce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * bce_loss
        return focal_loss.mean() if self.reduction == "mean" else focal_loss.sum()

# ======================
# 5️⃣ Custom Trainer
# ======================
class CustomTrainer(Trainer):
    def __init__(self, *args, alpha=1, gamma=2, **kwargs):
        super().__init__(*args, **kwargs)
        self.focal_loss = MultiLabelFocalLoss(alpha=alpha, gamma=gamma)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs["labels"]
        outputs = model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])
        logits = outputs.logits
        loss = self.focal_loss(logits, labels)
        return (loss, outputs) if return_outputs else loss

# ======================
# 6️⃣ Metrics
# ======================
def compute_metrics(pred):
    logits, labels = pred
    preds = (torch.sigmoid(torch.tensor(logits)) > 0.5).int().numpy()
    labels = labels
    return {
        "micro_f1": f1_score(labels, preds, average="micro", zero_division=0),
        "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
        "samples_f1": f1_score(labels, preds, average="samples", zero_division=0)
    }

# ======================
# 7️⃣ Training Args
# ======================
training_args = TrainingArguments(
    output_dir="./crime_model_multilabel",
    eval_strategy="epoch",   # ✅ fixed spelling
    save_strategy="epoch",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    learning_rate=5e-5,
    logging_dir="./logs",
    logging_steps=50,
    load_best_model_at_end=True,
    report_to=[],
    save_total_limit=2,
    fp16=torch.cuda.is_available()
)

# ======================
# 8️⃣ Trainer
# ======================
trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    alpha=1,
    gamma=2
)

# ======================
# 9️⃣ Train
# ======================
trainer.train()
trainer.save_model("./fine_tuned_crime_model_multilabel_focal")
tokenizer.save_pretrained("./fine_tuned_crime_model_multilabel_focal")


In [13]:
sampled_df.to_csv("Data.csv")

In [25]:
pd.concat([train_df, val_df]).to_csv("training_data.csv", index=False)
test_df.to_csv("testing_data.csv", index=False)

In [ ]:
from sklearn.metrics import classification_report, f1_score, accuracy_score
import torch
from tqdm import tqdm
import numpy as np

# ======================
# 0️⃣ Setup
# ======================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

categories = ["robbery", "suicide", "kidnapping", "terrorist attack", 
              "sexual assault", "murder", "other"]

# True labels (multi-hot vectors)
true_labels = np.array(test_dataset["labels"])

# ======================
# 1️⃣ Predict
# ======================
test_texts = [str(t) for t in test_dataset["text"]]
batch_size = 16
all_preds = []

for i in tqdm(range(0, len(test_texts), batch_size)):
    batch_texts = test_texts[i:i+batch_size]
    inputs = tokenizer(
        batch_texts,
        padding=True,
        truncation=True,
        max_length=512,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        # ✅ sigmoid for multi-label probabilities
        probs = torch.sigmoid(outputs.logits)
        preds = (probs > 0.5).int()   # threshold at 0.5
        all_preds.extend(preds.cpu().numpy())

preds = np.array(all_preds)

# ======================
# 2️⃣ Metrics
# ======================
micro_f1 = f1_score(true_labels, preds, average="micro", zero_division=0)
macro_f1 = f1_score(true_labels, preds, average="macro", zero_division=0)
samples_f1 = f1_score(true_labels, preds, average="samples", zero_division=0)

print(f"Micro F1:   {micro_f1:.4f}")
print(f"Macro F1:   {macro_f1:.4f}")
print(f"Samples F1: {samples_f1:.4f}")

# ======================
# 3️⃣ Classification report
# ======================
report = classification_report(true_labels, preds, target_names=categories, zero_division=0)
print("\nClassification Report:\n", report)


In [31]:
subset_acc = accuracy_score(true_labels, preds)

print(f"Subset Accuracy: {subset_acc:.4f}")

Subset Accuracy: 0.8240


In [28]:
from huggingface_hub import notebook_login

notebook_login()


In [40]:
save_path = "./fine_tuned_crime_model"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)


('./fine_tuned_crime_model/tokenizer_config.json',
 './fine_tuned_crime_model/special_tokens_map.json',
 './fine_tuned_crime_model/vocab.txt',
 './fine_tuned_crime_model/added_tokens.json',
 './fine_tuned_crime_model/tokenizer.json')

In [ ]:
from huggingface_hub import HfApi, HfFolder
from transformers import AutoTokenizer, AutoModelForSequenceClassification

repo_name = #Put your HF username/repo

# Push the model and tokenizer
model.push_to_hub(repo_name)
tokenizer.push_to_hub(repo_name)


In [ ]:
test_df.to_csv("hello.csv", index=False)

In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import classification_report, f1_score, accuracy_score
from sklearn.preprocessing import MultiLabelBinarizer
import ast  # for converting string lists to python lists
from tqdm import tqdm

# =========================
# 1️⃣ Load model from Hugging Face
# =========================
MODEL_ID = "the-usan/bert-crime-multilabel-eng"   # replace with your HF repo
categories = ["murder", "sexual assault", "kidnapping", "terrorist attack", "robbery", "suicide", "other"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("🔥 Using device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID)
model.to(device)
model.eval()

# =========================
# 2️⃣ Load your test file
# =========================
test_df = pd.read_csv("/kaggle/input/english-crime-data/hello.csv")

# If arr_labels column exists, convert to lists
if "arr_labels" in test_df.columns:
    test_df["arr_labels"] = test_df["arr_labels"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

texts = test_df["text"].astype(str).tolist()

# =========================
# 3️⃣ Run predictions in batches
# =========================
batch_size = 16  # adjust to fit GPU memory
all_predictions = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch_texts = texts[i:i+batch_size]
    inputs = tokenizer(batch_texts, padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.sigmoid(outputs.logits)

    batch_predicted_vectors = (probs > 0.5).int().cpu().tolist()
    all_predictions.extend(batch_predicted_vectors)

# Map back to categories
predictions = [[cat for cat, v in zip(categories, vec) if v == 1] for vec in all_predictions]
test_df["predicted_labels"] = predictions

# =========================
# 4️⃣ Evaluate (if labels exist)
# =========================
if "arr_labels" in test_df.columns:
    mlb = MultiLabelBinarizer(classes=categories)
    true_vectors = mlb.fit_transform(test_df["arr_labels"].tolist())

    # ✅ use all_predictions here, not predicted_vectors
    f1 = f1_score(true_vectors, all_predictions, average="micro")
    subset_acc = accuracy_score(true_vectors, all_predictions)
    report = classification_report(true_vectors, all_predictions, target_names=categories)

    print("🔹 Subset Accuracy:", subset_acc)
    print("🔹 Micro F1:", f1)
    print("🔹 Classification Report:\n", report)

# =========================
# 5️⃣ Save results
# =========================
test_df.to_csv("test_predictions.csv", index=False)
print("✅ Predictions saved to test_predictions.csv")
